# Italian NER with rules and spaCy

Use a **CPU runtime**. This notebook downloads and prepares the Italian corpus, then extracts entity candidates with rules and the Italian spaCy model. The result contains document text, candidate spans, and review flags.


## Set up the environment

The cell updates the checkout, prints its commit and installs the Italian spaCy model.


In [ ]:
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys

repo = Path("/content/forensic-llm-graph")
if not repo.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/AntoFratta/forensic-llm-graph.git", str(repo)],
        check=True,
    )
else:
    subprocess.run(["git", "pull", "--ff-only"], cwd=repo, check=True)

os.chdir(repo)
subprocess.run(["git", "log", "-1", "--oneline"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)


## Prepare the corpus

Upload the checked documents.json to /content/ to reuse the local preparation. The notebook can also download and prepare the registered sources.


In [ ]:
prepared_input = Path("/content/documents.json")
if prepared_input.is_file():
    destination = Path("data/processed/documents.json")
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(prepared_input, destination)
    print(f"Using {prepared_input}")
else:
    subprocess.run([sys.executable, "scripts/download_data.py"], check=True)
    subprocess.run([sys.executable, "scripts/prepare_data.py", "--overwrite"], check=True)


## Extract entity candidates

The script processes documents in batches and reports progress every 100 documents. The output keeps the original text and character offsets. Choose a new output path to keep an earlier run.


In [ ]:
run_number = "05"  # @param {type:"string"}
output_path = Path(f"outputs/{run_number}_ner_baseline.json")
command = [sys.executable, "-u", "scripts/run_pipeline.py", "--output", str(output_path)]

# Forward progress as the script processes the corpus.
with subprocess.Popen(
    command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
) as process:
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        return_code = process.wait()
    except KeyboardInterrupt:
        process.terminate()
        process.wait()
        raise

if return_code:
    raise subprocess.CalledProcessError(return_code, command)


## Inspect and save the output

Candidate counts describe what the extractor found. Precision, recall and F1 require human-reviewed annotations.

Download the numbered baseline JSON from the Files panel before ending the runtime. The second notebook takes this file as input.


In [ ]:
with output_path.open(encoding="utf-8") as result:
    run = json.load(result)["run"]

print(f"Model: {run['model_name']} {run['model_version']}")
print(f"Documents: {run['document_count']:,}")
print(f"Candidates: {run['candidate_count']:,}")
print(f"Extraction time: {run['extraction_seconds']:.1f} seconds")

for source, counts in run["by_source"].items():
    print(f"{source}: {counts['documents']:,} documents, {counts['candidates']:,} candidates")
